In [7]:
import sys
import copy
import tkinter as tk
from tkinter import ttk

sys.setrecursionlimit(100000)

board = [
    [2,8,3],
    [1,6,4],
    [7,0,5]
]

goal = [
    [1,2,3],
    [8,0,4],
    [7,6,5]
]

move = {
    (-1,0):"UP",
    (1,0):"DOWN",
    (0,-1):"LEFT",
    (0,1):"RIGHT"
}


def goal_test(state):
    return state == goal


def find_zero(state):

    for i in range(3):
        for j in range(3):

            if state[i][j]==0:
                return i,j


class node:

    def __init__(
        self,
        state_cur,
        parent_node,
        action,
        path_cost
    ):

        self.state_cur=state_cur
        self.parent_node=parent_node
        self.action=action
        self.path_cost=path_cost


def in_path(cur_node,state):

    while cur_node:

        if cur_node.state_cur==state:
            return True

        cur_node=cur_node.parent_node

    return False


def Depth_Limited_Search(
        problem,
        gioi_han
):

    stack=[]

    direction=[
        (0,1),
        (0,-1),
        (1,0),
        (-1,0)
    ]

    root=node(
        problem,
        None,
        None,
        0
    )

    stack.append(
        root
    )

    ket_qua="failure"

    while stack:

        cur_node=stack.pop()

        if goal_test(
            cur_node.state_cur
        ):
            return cur_node


        if cur_node.path_cost>=gioi_han:

            ket_qua="cutoff"

        else:

            x,y=find_zero(
                cur_node.state_cur
            )

            for dx,dy in direction:

                nx=x+dx
                ny=y+dy

                if(
                    0<=nx<3
                    and
                    0<=ny<3
                ):

                    new_state=copy.deepcopy(
                        cur_node.state_cur
                    )

                    new_state[x][y],new_state[nx][ny]=\
                    new_state[nx][ny],new_state[x][y]


                    if not in_path(
                        cur_node,
                        new_state
                    ):

                        child=node(
                            new_state,
                            cur_node,
                            move[(dx,dy)],
                            cur_node.path_cost+1
                        )

                        stack.append(
                            child
                        )

    return ket_qua


def Iterative_Deepening_Search(
        problem
):

    gioi_han=0

    while True:

        ket_qua=Depth_Limited_Search(
            problem,
            gioi_han
        )

        if ket_qua=="cutoff":

            gioi_han+=1

        elif ket_qua=="failure":

            return None

        else:

            return ket_qua


def trace_path(goal_node):

    path=[]

    cur=goal_node

    while cur:

        path.append(
            cur
        )

        cur=cur.parent_node

    path.reverse()

    return path


class PuzzleGUI:

    def __init__(self,root):

        self.root=root

        root.title(
            "8 Puzzle IDS"
        )

        top=tk.Frame(
            root
        )

        top.pack(
            pady=10
        )


        tk.Label(
            top,
            text="Algorithm:"
        ).pack(
            side="left"
        )


        self.combo=ttk.Combobox(
            top,
            values=[
                "BFS",
                "DFS",
                "IDS"
            ],
            state="readonly",
            width=10
        )

        self.combo.current(
            2
        )

        self.combo.pack(
            side="left",
            padx=5
        )


        tk.Button(
            top,
            text="Solve",
            command=self.start
        ).pack(
            side="left"
        )


        frame=tk.Frame(
            root
        )

        frame.pack(
            pady=20
        )


        self.start_board=\
        self.create_board(
            frame,
            0,
            "START"
        )

        self.current_board=\
        self.create_board(
            frame,
            1,
            "CURRENT"
        )

        self.goal_board=\
        self.create_board(
            frame,
            2,
            "GOAL"
        )


        self.draw(
            self.start_board,
            board
        )

        self.draw(
            self.goal_board,
            goal
        )


        self.info=tk.Label(
            root,
            font=(
                "Arial",
                12
            )
        )

        self.info.pack()



    def create_board(
            self,
            parent,
            column,
            title
    ):

        outer=tk.Frame(
            parent
        )

        outer.grid(
            row=0,
            column=column,
            padx=20
        )

        tk.Label(
            outer,
            text=title
        ).pack()

        grid=tk.Frame(
            outer
        )

        grid.pack()

        cells=[]

        for i in range(3):

            row=[]

            for j in range(3):

                lbl=tk.Label(
                    grid,
                    width=4,
                    height=2,
                    font=(
                        "Arial",
                        20
                    ),
                    relief="solid"
                )

                lbl.grid(
                    row=i,
                    column=j
                )

                row.append(
                    lbl
                )

            cells.append(
                row
            )

        return cells


    def draw(
            self,
            cells,
            state
    ):

        for i in range(3):

            for j in range(3):

                value=state[i][j]

                cells[i][j]["text"]=(
                    ""
                    if value==0
                    else str(value)
                )


    def start(self):

        algo=self.combo.get()

        if algo!="IDS":

            self.info["text"]=\
            "File này chỉ hỗ trợ IDS"

            return


        result=Iterative_Deepening_Search(
            board
        )


        if result is None:

            self.info["text"]="No solution"

            return


        self.path=trace_path(
            result
        )

        self.index=0

        self.animate()


    def animate(self):

        if self.index>=len(
                self.path
        ):

            self.info["text"]+="\nGoal Found"

            return


        cur=self.path[
            self.index
        ]


        self.draw(
            self.current_board,
            cur.state_cur
        )


        self.info["text"]=(
            f"Action: {cur.action}\n"
            f"Cost: {cur.path_cost}"
        )


        self.index+=1


        self.root.after(
            700,
            self.animate
        )


root=tk.Tk()

PuzzleGUI(root)

root.mainloop()